# Configs

Required Libraries: numpy, pandas, matplotlib, imageio, scipy

1. Toggle provided for saving gif (for visualization), saving pointclouds and enabling grid segmentation depending on available dataset
2. Tests were carried on Dell Inspiron 5420, 32 GB RAM, under nominal conditions with no backgrounds apps running
3. The INTERACTION dataset is proprietary and can be obtained from the official website by submitting the request form. Please refer to https://interaction-dataset.com/ for further details about the dataset and CSV format 

In [35]:
# Uncommend and run the following line to install dependencies if you haven't already

# !pip install numpy pandas matplotlib scipy imageio numba

# SPART

### Config Params and Libraries

In [20]:
import os
import math
import time
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Circle
import imageio
from typing import List, Tuple, Dict, Any
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import scipy.stats as sp_stats


plt.style.use('seaborn-v0_8-darkgrid')
np.set_printoptions(precision=4, suppress=True)

# Config - tune these
config = {
    "csv_path": "vehicle_tracks_000.csv",   # change to your CSV path
    "use_numba": True,
    "numba_threads": 4,                     # set to number of CPU threads to use
    "cell_size": 10.0,                      # grid cell size in meters (tune)
    "grid_padding": 5.0,                    # pad around dataset extent
    "frame_start": None,
    "frames_to_run": 8875,
    "fov_deg": 120.0,
    "delta_deg": 0.5,
    "r_max": 15.0,
    # Circular motion parameters for ego vehicle
    "ego_circular_motion": True,            # Enable circular motion
    "ego_circle_radius": 15.0,              # Radius of circular path in meters
    "ego_circle_speed": 5.398298934,        # Speed along circle in m/s
    "ego_circle_center": (996.0, 999.0),    # Center of circular path
    # Static ego parameters (used only if circular_motion is False)
    "ego_pos": (995.0, 1000.0),
    "ego_vel": (0.0, 0.0),
    "ego_psi": 0.0,
    "output_dir": "OutputSPART",
    "scans_dir": "OutputSPART/scans",
    "pc_dir": "OutputSPART/PointClouds",
    "save_gif": False,
    "gif_fps": 12,
    "frames_for_gif": 200,
    "warmup_numba": True,
    "enable_grid": False,
    "save_pointcloud": False           
}

# Collect environment information
env_info = {
    "platform": platform.platform(),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
}

os.makedirs(config["output_dir"], exist_ok=True)
os.makedirs(config["scans_dir"], exist_ok=True)
os.makedirs(config["pc_dir"], exist_ok=True)

# Check Numba availability and set threads
NUMBA_AVAILABLE = False
if config["use_numba"]:
    try:
        import numba
        from numba import njit, prange
        NUMBA_AVAILABLE = True
        try:
            numba.set_num_threads(config["numba_threads"])
        except Exception:
            pass
        print("Numba available - using Numba with threads =", numba.get_num_threads())
        env_info["numba_available"] = True
        env_info["numba_version"] = numba.__version__
        env_info["numba_threads"] = config["numba_threads"]
    except Exception as e:
        raise RuntimeError("Numba requested but not available. Install numba (pip install numba) and re-run.") from e
else:
    raise RuntimeError("This notebook is configured to use Numba-only. Set config['use_numba']=True and install numba.")
    env_info["numba_available"] = False
    env_info["numba_version"] = None

# Save benchmark config with environment info
with open(os.path.join(config["output_dir"], "benchmark_config.json"), "w") as f:
    json.dump({"config": config, "env_info": env_info}, f, indent=2)
print("Saved benchmark_config.json")

# Add hardware info saving function
def save_hardware_info(dst):
    """Save hardware and environment information to text file."""
    with open(dst, "w") as f:
        f.write("Environment Information\n")
        f.write("=" * 50 + "\n\n")
        for k, v in env_info.items():
            f.write(f"{k}: {v}\n")
        
        f.write("\n" + "=" * 50 + "\n")
        f.write("Hardware Information\n")
        f.write("=" * 50 + "\n\n")
        
        try:
            import psutil
            cpu = psutil.cpu_freq()
            mem = psutil.virtual_memory()
            
            f.write(f"CPU count (logical): {psutil.cpu_count(logical=True)}\n")
            f.write(f"CPU count (physical): {psutil.cpu_count(logical=False)}\n")
            if cpu:
                f.write(f"CPU frequency: {cpu.current:.2f} MHz\n")
            f.write(f"Total RAM: {mem.total/(1024**3):.2f} GB\n")
            f.write(f"Available RAM: {mem.available/(1024**3):.2f} GB\n")
        except ImportError:
            f.write("psutil not available - hardware details limited\n")
        except Exception as e:
            f.write(f"Error getting hardware info: {e}\n")

# Save hardware info
save_hardware_info(os.path.join(config["output_dir"], "hardware_info.txt"))
print("Saved hardware_info.txt")

Numba available - using Numba with threads = 4
Saved benchmark_config.json
Saved hardware_info.txt


### Function Definitions

In [21]:
EPS = 1e-9

def normalize_angle(a: float) -> float:
    """Normalize angle to (-pi, pi]"""
    return ((a + math.pi) % (2*math.pi)) - math.pi

def angle_diff(a: float, b: float) -> float:
    """Signed difference a - b in (-pi, pi]"""
    return normalize_angle(a - b)

def canonical_relative_angle(angle, psi):
    """
    Compute angle - psi in canonical range (-pi, pi].
    Use to discretize beam indices robustly.
    """
    return normalize_angle(angle - psi)

def cross2(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """2D cross product (vectorized if arrays)"""
    return a[...,0]*b[...,1] - a[...,1]*b[...,0]

# Memory tracking helper
def get_memory_mb():
    """Return RSS memory in MB (uses psutil if available)."""
    try:
        import psutil
        process = psutil.Process()
        return process.memory_info().rss / (1024**2)
    except:
        return float('nan')

# Ego Vehicle Circular Motion
def generate_autonomous_car_data(timestamp, radius=10, speed=5.398298934, center=np.array([996, 999])):
    """
    Generate ego vehicle state for circular motion.
    
    Parameters:
    -----------
    timestamp : float
        Current time in seconds
    radius : float
        Radius of circular path in meters
    speed : float
        Speed along the circle in m/s
    center : np.ndarray
        Center of circular path [x, y]
    
    Returns:
    --------
    dict with keys: 'x', 'y', 'vx', 'vy', 'psi_rad'
    """
    # Calculate the angular velocity needed for a circular motion
    angular_velocity = speed / radius  # rad/s
    
    # Calculate the current angle (theta) based on time and angular velocity
    theta = angular_velocity * timestamp  # angle in radians
    
    # Calculate the position on the circle at the current angle
    x = center[0] + radius * np.cos(theta)
    y = center[1] + radius * np.sin(theta)
    
    # Calculate the velocity components (tangential to the circle)
    vx = -speed * np.sin(theta)  # derivative of x with respect to time
    vy = speed * np.cos(theta)   # derivative of y with respect to time
    
    # Calculate the heading angle (psi) as the direction of the velocity vector
    psi_rad = np.arctan2(vy, vx)
    
    return {'x': x, 'y': y, 'vx': vx, 'vy': vy, 'psi_rad': psi_rad}

def get_vehicle_corners(center: Tuple[float,float], yaw: float, length: float, width: float) -> np.ndarray:
    """
    Return 4 corners in CCW order starting from front-right.
    Returns array shape (4,2).
    """
    cx, cy = center
    hl = length / 2.0
    hw = width / 2.0
    pts_local = np.array([
        [ hl, -hw],
        [ hl,  hw],
        [-hl,  hw],
        [-hl, -hw],
    ])  # (4,2)
    c = math.cos(yaw); s = math.sin(yaw)
    R = np.array([[c, -s],[s, c]])
    pts_world = (R @ pts_local.T).T + np.array([cx, cy])
    return pts_world

def get_segments_from_corners(corners: np.ndarray) -> np.ndarray:
    """Return array (4,2,2) segments from rectangle corners."""
    segs = []
    n = corners.shape[0]
    for i in range(n):
        p1 = corners[i]
        p2 = corners[(i+1) % n]
        segs.append(np.stack([p1, p2], axis=0))
    return np.array(segs)  # (4,2,2)

def build_segments_from_frame(frame_vehicles: List[Dict]) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return segments (M,2,2) and seg_track_ids (M,)
    Each vehicle contributes 4 segments (rectangle).
    """
    segs_list = []
    seg_track_ids = []
    for v in frame_vehicles:
        corners = get_vehicle_corners(v["pos"], v["yaw"], v["L"], v["W"])
        rect_segs = get_segments_from_corners(corners)  # (4,2,2)
        segs_list.append(rect_segs)
        seg_track_ids.extend([v["track_id"]] * rect_segs.shape[0])
    if len(segs_list) == 0:
        return np.zeros((0,2,2)), np.array([], dtype=int)
    segments = np.vstack(segs_list)
    return segments, np.array(seg_track_ids, dtype=int)

# spatial grid helpers (Python, outside numba)
def compute_dataset_bounds(frames_dict, padding=config["grid_padding"]):
    """Compute global bounding box of all vehicle centroids across frames."""
    xs = []; ys = []
    for fid, vehicles in frames_dict.items():
        for v in vehicles:
            x,y = v["pos"][0], v["pos"][1]
            xs.append(x); ys.append(y)
    if len(xs) == 0:
        raise RuntimeError("No vehicles found in frames.")
    xmin, xmax = min(xs)-padding, max(xs)+padding
    ymin, ymax = min(ys)-padding, max(ys)+padding
    return xmin, xmax, ymin, ymax

def create_grid(xmin, xmax, ymin, ymax, cell_size=config["cell_size"]):
    nx = max(1, int(math.ceil((xmax - xmin) / cell_size)))
    ny = max(1, int(math.ceil((ymax - ymin) / cell_size)))
    grid = {
        "xmin": xmin, "xmax": xmax, "ymin": ymin, "ymax": ymax,
        "cell_size": cell_size, "nx": nx, "ny": ny
    }
    return grid

def point_to_cell_ix(x, y, grid):
    i = int((x - grid["xmin"]) // grid["cell_size"])
    j = int((y - grid["ymin"]) // grid["cell_size"])
    # clamp
    if i < 0: i = 0
    if j < 0: j = 0
    if i >= grid["nx"]: i = grid["nx"]-1
    if j >= grid["ny"]: j = grid["ny"]-1
    return i, j

def bbox_to_cell_range(xmin_b, xmax_b, ymin_b, ymax_b, grid):
    i0, j0 = point_to_cell_ix(xmin_b, ymin_b, grid)
    i1, j1 = point_to_cell_ix(xmax_b, ymax_b, grid)
    return i0, i1, j0, j1

def build_frame_cell_index(frame_vehicles, grid):
    """
    Build mapping cell_index (i,j) -> list of vehicle indices for the given frame.
    We use centroids for assignment (fast).
    """
    cells = {}
    for idx, v in enumerate(frame_vehicles):
        x,y = float(v["pos"][0]), float(v["pos"][1])
        i,j = point_to_cell_ix(x,y,grid)
        key = (i,j)
        if key not in cells:
            cells[key] = []
        cells[key].append(idx)
    return cells

# sector bounding box helper
def sector_bbox(ego_pos, ego_psi, fov, r_max):
    """
    Compute axis-aligned bounding box (xmin,xmax,ymin,ymax) that contains the sector
    given by ego_pos, direction ego_psi, FOV fov (radians), and radius r_max.
    This is conservative (AABB) but fast for grid cell selection.
    """
    cx, cy = ego_pos[0], ego_pos[1]
    ang1 = ego_psi - fov/2.0
    ang2 = ego_psi + fov/2.0
    # sample the sector boundary at three points: ang1, ang2, and middle
    pts = []
    for a in [ang1, ang2, (ang1+ang2)/2.0]:
        pts.append((cx + r_max*math.cos(a), cy + r_max*math.sin(a)))
    xs = [p[0] for p in pts] + [cx]
    ys = [p[1] for p in pts] + [cy]
    return min(xs), max(xs), min(ys), max(ys)

# candidate selection combining grid + simple coarse checks
def get_candidate_indices(frame_vehicles, grid, ego_pos, ego_psi, fov, r_max):
    """
    Return list of vehicle indices (from frame_vehicles) that are candidates:
    - cells overlapping sector bounding box
    - coarse distance <= r_max*1.5 (optional slack)
    - coarse angular check of centroid relative to ego (within fov+slack)
    """
    xmin_b, xmax_b, ymin_b, ymax_b = sector_bbox(ego_pos, ego_psi, fov, r_max)
    i0, i1, j0, j1 = bbox_to_cell_range(xmin_b, xmax_b, ymin_b, ymax_b, grid)
    # collect vehicles in those cells
    cells_map = build_frame_cell_index(frame_vehicles, grid)
    cand_set = set()
    for i in range(i0, i1+1):
        for j in range(j0, j1+1):
            key = (i,j)
            if key in cells_map:
                for idx in cells_map[key]:
                    cand_set.add(idx)
    # apply coarse distance and angle filter
    candidates = []
    cx, cy = ego_pos[0], ego_pos[1]
    for idx in cand_set:
        v = frame_vehicles[idx]
        px, py = float(v["pos"][0]), float(v["pos"][1])
        dx = px - cx; dy = py - cy
        dist = math.hypot(dx, dy)
        # quick radial filter with 1.5 slack
        if dist > r_max*1.5:
            continue
        # coarse angular check: centroid angle relative to ego
        ang = math.atan2(dy, dx)
        rel = canonical_relative_angle(ang, ego_psi)
        if abs(rel) > (fov/2.0 + math.radians(2.0)):  # small slack 2 deg
            continue
        candidates.append(idx)
    # if no candidates found, fallback to checking nearby vehicles by distance to avoid misses
    if len(candidates) == 0:
        # fallback: include all vehicles within r_max*1.2
        for idx, v in enumerate(frame_vehicles):
            px, py = float(v["pos"][0]), float(v["pos"][1])
            dist = math.hypot(px - cx, py - cy)
            if dist <= r_max*1.2:
                candidates.append(idx)
    return candidates

# Numba core functions (parallel over beams)
from numba import njit, prange, float64, int64

@njit
def _norm_angle_numba(a):
    # normalize to (-pi,pi]
    return ((a + math.pi) % (2*math.pi)) - math.pi

@njit
def _segment_interval_numba(p1x, p1y, p2x, p2y, ego_x, ego_y):
    a1 = math.atan2(p1y - ego_y, p1x - ego_x)
    a2 = math.atan2(p2y - ego_y, p2x - ego_x)
    a1 = _norm_angle_numba(a1); a2 = _norm_angle_numba(a2)
    da = a2 - a1
    if da <= math.pi and da >= -math.pi:
        if da >= 0:
            amin = a1; amax = a2
        else:
            amin = a2; amax = a1
    else:
        if da >= 0:
            amin = a2; amax = a1
        else:
            amin = a1; amax = a2
    return amin, amax

@njit(parallel=True)
def simulate_scan_numba_core(ego_x, ego_y, ego_psi, fov, delta, p1s, p2s, r_max):
    """
    Numba-parallel over beams.
    p1s, p2s: arrays shape (M,2)
    returns angles (K), ranges (K), hit_segs (K), total_candidates (int)
    """
    half = fov / 2.0
    K = int(math.floor(fov / delta + 0.5)) + 1
    angles = np.empty(K, dtype=float64)
    for kk in range(K):
        angles[kk] = ego_psi - half + kk * delta

    M = p1s.shape[0]
    ranges = np.full(K, np.nan)
    hit_segs = np.full(K, -1, dtype=np.int64)
    total_candidates = 0

    # precompute seg endpoints arrays for numba access
    p1xs = np.empty(M, dtype=float64)
    p1ys = np.empty(M, dtype=float64)
    p2xs = np.empty(M, dtype=float64)
    p2ys = np.empty(M, dtype=float64)
    for i in range(M):
        p1xs[i] = p1s[i,0]; p1ys[i] = p1s[i,1]
        p2xs[i] = p2s[i,0]; p2ys[i] = p2s[i,1]

    # precompute angular intervals
    seg_amin = np.empty(M, dtype=float64)
    seg_amax = np.empty(M, dtype=float64)
    for i in range(M):
        amin, amax = _segment_interval_numba(p1xs[i], p1ys[i], p2xs[i], p2ys[i], ego_x, ego_y)
        seg_amin[i] = amin; seg_amax[i] = amax

    # per-beam loop (parallel)
    for k in prange(K):
        theta = angles[k]
        d0 = math.cos(theta); d1 = math.sin(theta)
        best_t = 1e99
        best_idx = -1
        # quickly test each segment's angular interval and intersection
        for i in range(M):
            amin = seg_amin[i]; amax = seg_amax[i]
            inside = False
            if amin <= amax:
                if theta >= amin and theta <= amax:
                    inside = True
            else:
                if theta >= amin or theta <= amax:
                    inside = True
            if not inside:
                continue
            # count candidate
            total_candidates += 1
            # compute intersection
            p1x = p1xs[i]; p1y = p1ys[i]
            p2x = p2xs[i]; p2y = p2ys[i]
            p1r0 = p1x - ego_x; p1r1 = p1y - ego_y
            sx = p2x - p1x; sy = p2y - p1y
            denom = d0*sy - d1*sx
            numer_t = p1r0*sy - p1r1*sx
            numer_u = p1r0*d1 - p1r1*d0
            if abs(denom) > 1e-9:
                t_i = numer_t / denom
                u_i = numer_u / denom
                if t_i >= 0.0 and u_i >= 0.0 and u_i <= 1.0 and t_i <= r_max:
                    if t_i < best_t:
                        best_t = t_i; best_idx = i
            else:
                # parallel or collinear
                if abs(numer_u) <= 1e-9:
                    t0 = p1r0 * d0 + p1r1 * d1
                    t1 = (p2x - ego_x) * d0 + (p2y - ego_y) * d1
                    tmin = t0 if t0 < t1 else t1
                    tmax = t1 if t1 > t0 else t0
                    if tmax >= 0.0 and tmin <= r_max:
                        t_hit = tmin if tmin >= 0.0 else 0.0
                        if t_hit < best_t:
                            best_t = t_hit; best_idx = i
        if best_idx != -1:
            ranges[k] = best_t
            hit_segs[k] = best_idx
    return angles, ranges, hit_segs, total_candidates

# wrapper simulate_scan that only uses Numba core
def simulate_scan_numba_wrapper(ego_pos, ego_psi, fov, delta, segments, seg_track_ids, r_max):
    """
    segments: (M,2,2)
    seg_track_ids: (M,)
    Calls the numba core which expects p1s and p2s arrays (M,2).
    Returns angles, ranges, hit_tracks (mapped to seg_track_ids), hit_segs (local index),
            and total_candidates count.
    """
    M = segments.shape[0]
    if M == 0:
        # build angles array for consistency
        half = fov/2.0
        K = int(math.floor(fov / delta + 0.5)) + 1
        angles = np.array([ego_psi - half + k*delta for k in range(K)])
        ranges = np.full(K, np.nan)
        hit_tracks = np.full(K, -1, dtype=int)
        hit_segs = np.full(K, -1, dtype=int)
        return angles, ranges, hit_tracks, hit_segs, 0

    p1s = segments[:,0,:].astype(np.float64)
    p2s = segments[:,1,:].astype(np.float64)
    angles, ranges, hit_segs, total_candidates = simulate_scan_numba_core(
        float(ego_pos[0]), float(ego_pos[1]), float(ego_psi),
        float(fov), float(delta), p1s, p2s, float(r_max)
    )
    # map hit_segs to track ids
    hit_tracks = np.full_like(hit_segs, -1)
    for k in range(hit_segs.shape[0]):
        si = int(hit_segs[k])
        if si >= 0:
            hit_tracks[k] = int(seg_track_ids[si])
    return angles, ranges, hit_tracks, hit_segs, int(total_candidates)

# estimate_next_revisit_time (as in prior code, keep numerically stable)
def estimate_next_revisit_time(ego_pos: np.ndarray, ego_vel: np.ndarray,
                               veh_pos: np.ndarray, veh_vel: np.ndarray,
                               r: float, tcurr: float, eps: float = 1e-6):
    rel = veh_pos - ego_pos
    dist = np.linalg.norm(rel)
    if dist <= r + eps:
        return tcurr
    dir_unit = rel / (dist + eps)
    v_rel_along = np.dot(veh_vel - ego_vel, dir_unit)
    if v_rel_along <= eps:
        return tcurr + 0.2
    t_est = (dist - r) / v_rel_along
    if t_est < 0:
        t_est = 0.0
    return tcurr + float(t_est)

def process_frame_with_grid(frame_vehicles: List[Dict], grid, enable_grid,
                            ego_pos: Tuple[float,float], ego_vel: Tuple[float,float],
                            ego_psi: float, fov: float, delta: float, r_max: float,
                            tcurr: float, eligibility: Dict[int, float]):
    """
    Process a single frame using:
      - temporal muting (eligibility): skip vehicles with next_time > tcurr
      - grid selection: coarse candidate vehicles by spatial grid & sector bbox
      - build geometry (segments) for selected vehicles only
      - call Numba scan
    Returns: angles, ranges, hit_tracks, hit_segs, metrics dict
    """
    # 1) select vehicles by eligibility (still must consider skipped vehicles for eligibility updates)
    process_vehicles = []
    selected_vehicles = []
    skip_vehicles = []
    for v in frame_vehicles:
        tid = v["track_id"]
        next_t = eligibility.get(tid, 0.0)
        if tcurr >= next_t:
            process_vehicles.append(v)
        else:
            skip_vehicles.append(v)

    # 2) coarse grid-based candidate selection among process_vehicles
    # if few process_vehicles, skip grid to avoid overhead
    if len(process_vehicles) == 0:
        segments = np.zeros((0,2,2), dtype=float)
        seg_track_ids = np.array([], dtype=int)
    elif enable_grid:
        # we still want to limit to those likely in FOV/radius with grid
        cand_idxs = get_candidate_indices(process_vehicles, grid, ego_pos, ego_psi, fov, r_max)
        # build a filtered vehicle list
        selected_vehicles = [process_vehicles[i] for i in cand_idxs]
        segments, seg_track_ids = build_segments_from_frame(selected_vehicles)
    else:
        segments, seg_track_ids = build_segments_from_frame(process_vehicles)

    t0 = time.perf_counter()
    angles, ranges, hit_tracks, hit_segs, total_candidates = simulate_scan_numba_wrapper(
        ego_pos, ego_psi, fov, delta, segments, seg_track_ids, r_max
    )
    t1 = time.perf_counter()
    time_scan_sec = t1 - t0

    # Update eligibility for processed vehicles (we must update for those we attempted to process,
    # but note we only built geometry for selected_vehicles; we still mark processed_vehicles as processed)
    for v in process_vehicles:
        tid = v["track_id"]
        eligibility[tid] = estimate_next_revisit_time(np.array(ego_pos), np.array(ego_vel),
                                                      v["pos"], v["vel"], r_max, tcurr)
    for v in skip_vehicles:
        eligibility[v["track_id"]] = eligibility.get(v["track_id"], tcurr + 1.0)

    # Measure memory
    mem_mb = get_memory_mb()

    metrics = {
        "time_scan_sec": time_scan_sec,
        "n_vehicles": len(frame_vehicles),
        "n_processed": len(process_vehicles),
        "n_candidates": int(total_candidates),
        "n_segments": int(segments.shape[0]),
        "K_beams": int(np.floor(fov / delta + 0.5)) + 1,
        "num_hits": int(np.sum(~np.isnan(ranges))),
        "memory_rss_mb": mem_mb
    }
    return angles, ranges, hit_tracks, hit_segs, metrics, segments, seg_track_ids


def draw_scene_pub(ax, ego_pos, ego_psi, frame_vehicles=None, angles=None, ranges=None,
                   r_max=None, fov=None, timestamp=None, scale_bar_len=10.0):
    ax.clear()
    ax.set_aspect('equal', 'box')
    ax.grid(False)  # avoid heavy grids; prefer light or no grid for publication
    ax.scatter([ego_pos[0]], [ego_pos[1]], s=36, marker='o', color='k', zorder=5, label='Ego')
    # heading arrow: Fancy arrow with white edge for clarity
    ax.arrow(ego_pos[0], ego_pos[1], 0.8*math.cos(ego_psi), 0.8*math.sin(ego_psi),
             head_width=0.25, head_length=0.3, color='k', linewidth=1.0, zorder=6)

    # draw vehicles with light fill and colored edge according to status
    handles = {}  # for legend
    if frame_vehicles is not None:
        for v in frame_vehicles:
            corners = get_vehicle_corners(v["pos"], v["yaw"], v["L"], v["W"])
            rel = corners - np.array([ego_pos[0], ego_pos[1]])
            dists = np.hypot(rel[:,0], rel[:,1])
            inside_radius_any = np.any(dists <= (r_max + 1e-9)) if r_max is not None else False
            corner_angles = np.arctan2(rel[:,1], rel[:,0])
            rel_angles = canonical_relative_angle(corner_angles, ego_psi)
            in_fov_any = np.any(np.abs(rel_angles) <= (fov/2.0 + 1e-9)) if fov is not None else False

            if inside_radius_any:
                edge_col = 'green' if in_fov_any else 'red'
            else:
                edge_col = 'black'

            # subtle fill, thin edge
            poly = Polygon(corners, closed=True, facecolor='0.92', edgecolor=edge_col,
                           linewidth=1.0, zorder=3)
            ax.add_patch(poly)

            # optional small text ID
            if 'id' in v:
                cx, cy = np.mean(corners, axis=0)
                ax.text(cx, cy, str(v['id']), fontsize=6, ha='center', va='center', zorder=7)

            # create legend handles only once per color/type
            key = ('in_fov' if (inside_radius_any and in_fov_any) else
                   'in_range_out_fov' if inside_radius_any else 'out_range')
            if key not in handles:
                if key == 'in_fov':
                    handles[key] = mpatches.Patch(edgecolor='green', facecolor='0.92', label='In range and FOV')
                elif key == 'in_range_out_fov':
                    handles[key] = mpatches.Patch(edgecolor='red', facecolor='0.92', label='In range, not FOV')
                else:
                    handles[key] = mpatches.Patch(edgecolor='black', facecolor='0.92', label='Out of range')

    if angles is not None and ranges is not None:
        for k, theta in enumerate(angles):
            rr = ranges[k]
            x_end = ego_pos[0] + (r_max if np.isnan(rr) else rr) * math.cos(theta)
            y_end = ego_pos[1] + (r_max if np.isnan(rr) else rr) * math.sin(theta)
            ax.plot([ego_pos[0], x_end], [ego_pos[1], y_end], '-', linewidth=0.4, alpha=0.18, zorder=1)
            if not np.isnan(rr):
                hx = ego_pos[0] + rr * math.cos(theta)
                hy = ego_pos[1] + rr * math.sin(theta)
                ax.scatter([hx], [hy], s=5, marker='o', facecolor='none', edgecolor='red', linewidth=0.8, zorder=4)

    if r_max is not None:
        circle = Circle((ego_pos[0], ego_pos[1]), r_max, fill=False, linestyle='--', linewidth=0.8, alpha=0.7, zorder=2)
        ax.add_patch(circle)
    if fov is not None and r_max is not None:
        x1 = ego_pos[0] + r_max * math.cos(ego_psi - fov/2)
        y1 = ego_pos[1] + r_max * math.sin(ego_psi - fov/2)
        x2 = ego_pos[0] + r_max * math.cos(ego_psi + fov/2)
        y2 = ego_pos[1] + r_max * math.sin(ego_psi + fov/2)
        ax.plot([ego_pos[0], x1], [ego_pos[1], y1], '-', linewidth=0.9, zorder=2)
        ax.plot([ego_pos[0], x2], [ego_pos[1], y2], '-', linewidth=0.9, zorder=2)
        # annotate FOV angle
        ax.text(ego_pos[0] + 0.5*r_max*math.cos(ego_psi), ego_pos[1] + 0.5*r_max*math.sin(ego_psi),
                f'FOV={np.degrees(fov):.0f}°', fontsize=7, ha='center', va='center', zorder=8)

    if handles:
        ax.legend(handles=list(handles.values()), loc='upper right', frameon=False)

    if timestamp is not None:
        ax.text(0.99, 0.01, timestamp, transform=ax.transAxes, ha='right', va='bottom', fontsize=7)

    # scale bar (bottom-left)
    sb_x = 0.02; sb_y = 0.02
    sb_xdata = ax.get_xlim()[0] + sb_x*(ax.get_xlim()[1]-ax.get_xlim()[0])
    sb_ydata = ax.get_ylim()[0] + sb_y*(ax.get_ylim()[1]-ax.get_ylim()[0])
    ax.plot([sb_xdata, sb_xdata + scale_bar_len], [sb_ydata, sb_ydata], '-', linewidth=1.0, solid_capstyle='butt')
    ax.text(sb_xdata + scale_bar_len/2, sb_ydata - 0.02*(ax.get_ylim()[1]-ax.get_ylim()[0]),
            f'{scale_bar_len:.0f} m', ha='center', va='top', fontsize=7)

    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.set_xlim(965, 1025); ax.set_ylim(965, 1025)


def save_gif_from_frames(frame_fns, out_path='spart_scan.gif', fps=8):
    imgs = [imageio.v2.imread(fn) for fn in frame_fns]
    imageio.mimsave(out_path, imgs, fps=fps)
    print("Saved GIF:", out_path)

def save_pointcloud_csv(frame_id, angles, ranges, out_prefix="pc_frame"):
    valid_idx = ~np.isnan(ranges)
    theta_deg = np.degrees(angles[valid_idx])
    rvals = ranges[valid_idx]
    df = pd.DataFrame({"theta_deg": theta_deg, "range_m": rvals})
    fn = os.path.join(config["pc_dir"], f"{out_prefix}_{frame_id:06d}.csv")
    df.to_csv(fn, index=False)
    return fn


### Dataset Loading and Preparation

In [17]:
# load CSV and optionally precompute grid extents
csv_path = config["csv_path"]
if not os.path.exists(csv_path):
    raise RuntimeError(f"CSV dataset not found at {csv_path}. Please set config['csv_path'] correctly.")

df = pd.read_csv(csv_path)
df = df.sort_values(["frame_id", "track_id"]).reset_index(drop=True)

# build frames dict
frames = {}
frame_timestamps = {}
for frame_id, group in df.groupby("frame_id"):
    vehicles = []
    for _, row in group.iterrows():
        vehicles.append({
            "track_id": int(row.track_id),
            "pos": np.array([row.x, row.y], dtype=float),
            "vel": np.array([row.vx, row.vy], dtype=float),
            "yaw": float(row.psi_rad),
            "L": float(row.length),
            "W": float(row.width),
            "speed": float(row.speed)
        })
    frames[int(frame_id)] = vehicles
    frame_timestamps[int(frame_id)] = float(group.timestamp_ms.iloc[0]) / 1000.0

print(f"Loaded {len(frames)} frames")

grid = None
# compute grid extents from dataset
if config["enable_grid"]:
    xmin, xmax, ymin, ymax = compute_dataset_bounds(frames, padding=config["grid_padding"])
    grid = create_grid(xmin, xmax, ymin, ymax, cell_size=config["cell_size"])
    print("Created spatial grid:", grid)


start_frame = min(frames.keys()) if config["frame_start"] is None else config["frame_start"]
end_frame = start_frame + config["frames_to_run"] - 1
frame_list = [f for f in range(start_frame, end_frame+1) if f in frames]


Loaded 8875 frames


### SPART Core Execution

In [24]:
eligibility = {}
results = []
gif_frames = []
for i, fid in enumerate(frame_list):
    fv = frames[fid]
    tcurr = frame_timestamps.get(fid, float(fid))
    

    if config["ego_circular_motion"]:
        ego_state = generate_autonomous_car_data(
            timestamp=tcurr,
            radius=config["ego_circle_radius"],
            speed=config["ego_circle_speed"],
            center=np.array(config["ego_circle_center"])
        )
        ego_pos = (ego_state['x'], ego_state['y'])
        ego_vel = (ego_state['vx'], ego_state['vy'])
        ego_psi = ego_state['psi_rad']
    else:
        # Use static values from config
        ego_pos = config["ego_pos"]
        ego_vel = config["ego_vel"]
        ego_psi = config["ego_psi"]
    # ============================================================
    
    angles, ranges, hit_tracks, hit_segs, metrics, segments, seg_track_ids = process_frame_with_grid(
        fv, grid, config["enable_grid"], ego_pos, ego_vel, ego_psi,
        math.radians(config["fov_deg"]), math.radians(config["delta_deg"]),
        config["r_max"], tcurr, eligibility
    )
    
    # logging
    row = {
        "frame_id": fid, "timestamp_s": tcurr,
        "ego_x": ego_pos[0], "ego_y": ego_pos[1], "ego_psi": ego_psi,
        "N_vehicles": metrics["n_vehicles"],
        "P_processed": metrics["n_processed"],
        "S_segments": metrics["n_segments"],
        "K_beams": metrics["K_beams"],
        "sum_m_k": metrics["n_candidates"],
        "num_hits": metrics["num_hits"],
        "time_scan_sec": metrics["time_scan_sec"],
        "memory_rss_mb": metrics["memory_rss_mb"]
    }
    results.append(row)

    # save a sample scan image
    if config["save_gif"]:
        fn = os.path.join(config["scans_dir"], f"scan_{i:04d}.png")
        fig, ax = plt.subplots(figsize=(6,6))
        draw_scene_pub(
            ax,
            ego_pos,
            ego_psi,
            frame_vehicles=fv,
            angles=angles,
            ranges=ranges,
            r_max=config["r_max"],
            fov=math.radians(config["fov_deg"]),
            timestamp=f"t = {tcurr:.2f} s",
            scale_bar_len=10.0
        )
        fig.savefig(fn, dpi=120)
        plt.close(fig)
        gif_frames.append(fn)

    if (i % 20) == 0:
        print(f"[{i}/{len(frame_list)}] fid={fid} time={metrics['time_scan_sec']*1000:.2f} ms, "
              f"ego=({ego_pos[0]:.1f}, {ego_pos[1]:.1f}), psi={np.degrees(ego_psi):.1f}°, "
              f"vehicles={metrics['n_vehicles']}, seg={metrics['n_segments']}, candidates={metrics['n_candidates']}")
    if config["save_pointcloud"]:
        save_pointcloud_csv(fid, angles, ranges) 

[0/8750] fid=0 time=0.27 ms, ego=(1011.0, 999.0), psi=90.0°, vehicles=10, seg=40, candidates=6
[20/8750] fid=20 time=0.15 ms, ego=(1007.3, 1008.9), psi=131.2°, vehicles=8, seg=12, candidates=52
[40/8750] fid=40 time=0.05 ms, ego=(998.0, 1013.9), psi=172.5°, vehicles=7, seg=16, candidates=0
[60/8750] fid=60 time=0.04 ms, ego=(987.7, 1011.5), psi=-146.3°, vehicles=6, seg=8, candidates=46
[80/8750] fid=80 time=0.05 ms, ego=(981.5, 1002.9), psi=-105.0°, vehicles=6, seg=12, candidates=100
[100/8750] fid=100 time=0.09 ms, ego=(982.5, 992.4), psi=-63.8°, vehicles=9, seg=24, candidates=0
[120/8750] fid=120 time=0.08 ms, ego=(990.2, 985.1), psi=-22.6°, vehicles=7, seg=8, candidates=0
[140/8750] fid=140 time=0.09 ms, ego=(1000.8, 984.8), psi=18.7°, vehicles=9, seg=16, candidates=100
[160/8750] fid=160 time=0.14 ms, ego=(1009.0, 991.5), psi=59.9°, vehicles=11, seg=28, candidates=610
[180/8750] fid=180 time=0.05 ms, ego=(1010.7, 1001.9), psi=101.2°, vehicles=11, seg=24, candidates=0
[200/8750] fid

### Logging and Metric Computation

In [25]:
# Save logs
df_out = pd.DataFrame(results)
df_out.to_csv(os.path.join(config["output_dir"], "per_frame_log.csv"), index=False)
print("Saved per-frame log:", os.path.join(config["output_dir"], "per_frame_log.csv"))

# Save GIF
if config["save_gif"] and len(gif_frames) > 0:
    out_gif = os.path.join(config["output_dir"], "spart_scan.gif")
    save_gif_from_frames(gif_frames[:config["frames_for_gif"]], out_path=out_gif, fps=config["gif_fps"])

# summary metrics + simple plots
import matplotlib
matplotlib.use('Agg')  # works in headless servers for saving

df_results = pd.read_csv(os.path.join(config["output_dir"], "per_frame_log.csv"))

# Calculate derived columns
df_results["time_ms"] = df_results["time_scan_sec"] * 1000.0
df_results["M_total"] = df_results["N_vehicles"] * 4  # 4 segments per vehicle
df_results["K_times_M"] = df_results["K_beams"] * df_results["M_total"]
df_results["SEI_frame"] = df_results["K_times_M"] / df_results["sum_m_k"].replace(0, np.nan)
df_results["eligibility_muted_ratio"] = (
    (df_results["N_vehicles"] - df_results["P_processed"]) / 
    df_results["N_vehicles"].replace(0, np.nan)
)
print(np.shape(df_results))
# Simple aggregates for summary.json
summary = {
    "frames": len(df_results),
    "mean_time_ms": df_results["time_ms"].mean(),
    "median_time_ms": df_results["time_ms"].median(),
    "std_time_ms": df_results["time_ms"].std(),
    "95th_time_ms": df_results["time_ms"].quantile(0.95),
    "fps_mean": 1000.0 / df_results["time_ms"].mean(),
    "mean_sum_m_k": df_results["sum_m_k"].mean(),
    "mean_memory_mb": df_results["memory_rss_mb"].mean(skipna=True),
    "mean_SEI": df_results["SEI_frame"].mean(skipna=True),
    "mean_eligibility_muted_ratio": df_results["eligibility_muted_ratio"].mean(skipna=True)
}

with open(os.path.join(config["output_dir"], "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Summary:", summary)

# Create comprehensive summary_stats.json
summary_stats = {
    "frames": len(df_results),
    "mean_time_ms": df_results["time_ms"].mean(),
    "median_time_ms": df_results["time_ms"].median(),
    "fps_mean": 1000.0 / df_results["time_ms"].mean(),
    "mean_sum_m_k": df_results["sum_m_k"].mean(),
    "mean_SEI": df_results["SEI_frame"].mean(skipna=True),
    "mean_eligibility_muted_ratio": df_results["eligibility_muted_ratio"].mean(skipna=True),
    "mean_memory_mb": df_results["memory_rss_mb"].mean(skipna=True)
}

with open(os.path.join(config["output_dir"], "summary_stats.json"), "w") as f:
    json.dump(summary_stats, f, indent=2)
print("Saved summary_stats.json")

Saved per-frame log: OutputSPART\per_frame_log.csv
(8750, 18)
Summary: {'frames': 8750, 'mean_time_ms': np.float64(0.0834578742480921), 'median_time_ms': np.float64(0.0774000000092201), 'std_time_ms': np.float64(0.04065339896137794), '95th_time_ms': np.float64(0.1443550005205249), 'fps_mean': np.float64(11982.092870318473), 'mean_sum_m_k': np.float64(104.70582857142857), 'mean_memory_mb': np.float64(2282.261666517857), 'mean_SEI': np.float64(115.67477146574582), 'mean_eligibility_muted_ratio': np.float64(0.5410833524831844)}
Saved summary_stats.json


# Vanilla RayCasting (Benchmark)

### Config Params and Libraries

In [26]:
import math
import time
import os
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import imageio
from typing import List, Tuple, Dict, Any

plt.style.use('seaborn-v0_8-darkgrid')
np.set_printoptions(precision=4, suppress=True)

try:
    import psutil
    PSUTIL_AVAILABLE = True
except Exception:
    PSUTIL_AVAILABLE = False

# Configuration - edit as needed
config = {
    "csv_path": "vehicle_tracks_000.csv",
    "frame_start": 0,
    "frame_end": None,
    "frames_to_run": 8874,
    "fov_deg": 120.0,
    "delta_deg": 0.5,
    "r_max": 15.0,
    "output_dir": "OutputVanilla",
    "plots_dir": "OutputVanilla/plots",
    "save_plots_svg": False,
    "enable_instrumentation": True,
    "random_seed": 12345,
    "synthetic_densities": [10,25,50,100,200],
    "synthetic_repeats": 3
}

# create directories
os.makedirs(config["output_dir"], exist_ok=True)
os.makedirs(config["plots_dir"], exist_ok=True)

# record environment info
env_info = {
    "platform": platform.platform(),
    "python_version": platform.python_version(),
    "numpy": np.__version__,
    "psutil_available": PSUTIL_AVAILABLE
}
with open(os.path.join(config["output_dir"], "benchmark_config.json"), "w") as f:
    json.dump({"config": config, "env_info": env_info}, f, indent=2)

### Function Definitions

In [27]:

def get_memory_mb():
    if PSUTIL_AVAILABLE:
        proc = psutil.Process()
        return proc.memory_info().rss / (1024**2)
    else:
        try:
            import tracemalloc
            cur, peak = tracemalloc.get_traced_memory()
            return peak / (1024**2)
        except Exception:
            return float('nan')


def normalize_angle(a: float) -> float:
    return ((a + np.pi) % (2*np.pi)) - np.pi

def cross2(a, b):
    # a and b are 2-element iterables or numpy arrays
    return a[0]*b[1] - a[1]*b[0]

def get_vehicle_corners(center: Tuple[float,float], yaw: float, length: float, width: float) -> np.ndarray:
    cx, cy = center
    hl = length / 2.0
    hw = width / 2.0
    pts_local = np.array([
        [ hl, -hw],
        [ hl,  hw],
        [-hl,  hw],
        [-hl, -hw],
    ])
    c = math.cos(yaw); s = math.sin(yaw)
    R = np.array([[c, -s],[s, c]])
    pts_world = (R @ pts_local.T).T + np.array([cx, cy])
    return pts_world

def get_segments_from_corners(corners: np.ndarray) -> List[Tuple[np.ndarray, np.ndarray]]:
    segs = []
    n = corners.shape[0]
    for i in range(n):
        p1 = corners[i]
        p2 = corners[(i+1) % n]
        segs.append((p1.copy(), p2.copy()))
    return segs


def ray_segment_intersection_simple(ego_pos, ray_dir, p1, p2, r_max, eps=1e-9):
    ox, oy = float(ego_pos[0]), float(ego_pos[1])
    dx, dy = float(ray_dir[0]), float(ray_dir[1])
    p1x, p1y = float(p1[0]), float(p1[1])
    p2x, p2y = float(p2[0]), float(p2[1])
    sx = p2x - p1x
    sy = p2y - p1y
    rx = dx; ry = dy
    denom = rx * sy - ry * sx
    px = p1x - ox; py = p1y - oy
    if abs(denom) > eps:
        t = (px * sy - py * sx) / denom
        u = (px * ry - py * rx) / denom
        if t >= 0.0 and u >= 0.0 and u <= 1.0 and t <= r_max:
            return float(t)
        else:
            return None
    else:
        col = abs(px * ry - py * rx) <= eps
        if not col:
            return None
        t0 = px * rx + py * ry
        t1 = (p2x - ox) * rx + (p2y - oy) * ry
        tmin = min(t0, t1); tmax = max(t0, t1)
        if tmax < 0 or tmin > r_max:
            return None
        return float(max(0.0, tmin))

def simulate_scan_vanilla_instrumented(ego_pos: Tuple[float,float], ego_psi: float,
                                       fov: float, delta: float,
                                       segments: List[Tuple[np.ndarray,np.ndarray]],
                                       seg_track_ids: List[int], r_max: float,
                                       enable_instrumentation: bool = True):

    t_start = time.perf_counter()
    half = fov / 2.0
    angles = []
    th = ego_psi - half
    while th <= ego_psi + half + 1e-12:
        angles.append(th)
        th += delta
    angles = np.array(angles)
    K = angles.size
    ranges = np.full(K, np.nan)
    hit_tracks = np.full(K, -1, dtype=int)

    total_candidates = 0
    # If no segments, trivial
    if len(segments) == 0:
        t_end = time.perf_counter()
        return angles, ranges, hit_tracks, 0, float(t_end - t_start)

    # core nested loops
    for k in range(K):
        theta = angles[k]
        d = np.array([math.cos(theta), math.sin(theta)])
        best_t = None
        best_tid = -1
        for si, (p1, p2) in enumerate(segments):
            # count candidate
            if enable_instrumentation:
                total_candidates += 1
            t = ray_segment_intersection_simple(ego_pos, d, p1, p2, r_max)
            if t is not None:
                if best_t is None or t < best_t:
                    best_t = t
                    best_tid = seg_track_ids[si]
        if best_t is not None:
            ranges[k] = best_t
            hit_tracks[k] = int(best_tid)

    t_end = time.perf_counter()
    return angles, ranges, hit_tracks, int(total_candidates), float(t_end - t_start)


def build_segments_list_flat(frame_vehicles: List[Dict]):
    segments = []
    seg_track_ids = []
    for v in frame_vehicles:
        corners = get_vehicle_corners(v["pos"], v["yaw"], v["L"], v["W"])
        segs = get_segments_from_corners(corners)
        for (p1, p2) in segs:
            segments.append((p1, p2))
            seg_track_ids.append(int(v["track_id"]))
    return segments, seg_track_ids

def process_frame_vanilla_instrumented(frame_vehicles: List[Dict], ego_pos: Tuple[float,float], ego_psi: float,
                                       fov: float, delta: float, r_max: float, enable_instrumentation=True):
    t_build0 = time.perf_counter()
    segments, seg_track_ids = build_segments_list_flat(frame_vehicles)
    t_build1 = time.perf_counter()
    time_build = t_build1 - t_build0

    angles, ranges, hit_tracks, total_candidates, time_total = simulate_scan_vanilla_instrumented(
        ego_pos, ego_psi, fov, delta, segments, seg_track_ids, r_max, enable_instrumentation=enable_instrumentation)

    mem_mb = get_memory_mb() if enable_instrumentation else float('nan')

    metrics = {
        "time_build_sec": time_build,
        "time_total_sec": time_total,
        "n_vehicles": len(frame_vehicles),
        "n_segments": len(segments),
        "K_beams": angles.size,
        "sum_m_k": int(total_candidates),
        "num_hits": int(np.sum(~np.isnan(ranges))),
        "memory_rss_mb": mem_mb
    }
    return angles, ranges, hit_tracks, metrics

csv_path = config["csv_path"]
df = pd.read_csv(csv_path)
df = df.sort_values(["frame_id", "track_id"]).reset_index(drop=True)
frames = {}
frame_timestamps = {}
for frame_id, group in df.groupby("frame_id"):
    vehicles = []
    for _, row in group.iterrows():
        vehicles.append({
            "track_id": int(row.track_id),
            "pos": np.array([row.x, row.y], dtype=float),
            "vel": np.array([row.vx, row.vy], dtype=float),
            "yaw": float(row.psi_rad),
            "L": float(row.length),
            "W": float(row.width),
            "speed": float(row.speed)
        })
    frames[int(frame_id)] = vehicles
    frame_timestamps[int(frame_id)] = float(group.timestamp_ms.iloc[0]) / 1000.0

all_frames_sorted = sorted(frames.keys())
if config["frame_start"] is None:
    config["frame_start"] = all_frames_sorted[0]
if config["frame_end"] is None:
    config["frame_end"] = config["frame_start"] + config["frames_to_run"] - 1
print(f"Loaded {len(frames)} frames. Running frames {config['frame_start']}..{config['frame_end']}")

def draw_scene(ax, ego_pos, ego_psi, segments, seg_track_ids, angles=None, ranges=None, r_max=None, fov=None):
    ax.clear(); ax.set_aspect('equal', 'box')
    ax.grid(True, alpha=0.3)
    ax.plot(ego_pos[0], ego_pos[1], 'ko', markersize=6)
    ax.arrow(ego_pos[0], ego_pos[1], 0.8*math.cos(ego_psi), 0.8*math.sin(ego_psi),
             head_width=0.2, head_length=0.25, color='k')  
    # draw rectangles as polygons
    # segments array is (M,2,2), but to draw rectangles nicely we group by track id
    if segments.shape[0] > 0:
        # group segments by track id and collect their corner points
        unique_tracks = np.unique(seg_track_ids)
        # print("Unique tracks in scene:", unique_tracks)
        for tid in unique_tracks:
            idxs = np.where(seg_track_ids == tid)[0]
            # collect unique vertices for this track from segments
            pts = []
            for si in idxs:
                p1 = segments[si,0]; p2 = segments[si,1]
                pts.append(tuple(p1)); pts.append(tuple(p2))
            # find convex hull (simple approx: take unique and create polygon)
            pts_unique = list(dict.fromkeys(pts))
            if len(pts_unique) >= 3:
                poly = Polygon(pts_unique, closed=True, alpha=0.4, edgecolor='black')
                ax.add_patch(poly)
            else:
                # draw segments
                for si in idxs:
                    p1 = segments[si,0]; p2 = segments[si,1]
                    ax.plot([p1[0], p2[0]],[p1[1], p2[1]], '-', linewidth=2, alpha=0.7)

    # draw beams and hits
    if angles is not None and ranges is not None:
        for k, theta in enumerate(angles):
            rr = ranges[k]
            x_end = ego_pos[0] + (r_max if np.isnan(rr) else rr) * math.cos(theta)
            y_end = ego_pos[1] + (r_max if np.isnan(rr) else rr) * math.sin(theta)
            ax.plot([ego_pos[0], x_end], [ego_pos[1], y_end], '-', linewidth=0.4, alpha=0.25)
            if not np.isnan(rr):
                hx = ego_pos[0] + rr * math.cos(theta)
                hy = ego_pos[1] + rr * math.sin(theta)
                ax.scatter([hx], [hy], s=8, c='red')

    if fov is not None and r_max is not None:
        # draw range circle
        circle = plt.Circle((ego_pos[0], ego_pos[1]), r_max, fill=False, linestyle='--', alpha=0.4)
        ax.add_patch(circle)
        # FOV boundaries
        x1 = ego_pos[0] + r_max * math.cos(ego_psi - fov/2)
        y1 = ego_pos[1] + r_max * math.sin(ego_psi - fov/2)
        x2 = ego_pos[0] + r_max * math.cos(ego_psi + fov/2)
        y2 = ego_pos[1] + r_max * math.sin(ego_psi + fov/2)
        ax.plot([ego_pos[0], x1], [ego_pos[1], y1], '-', linewidth=1.2)
        ax.plot([ego_pos[0], x2], [ego_pos[1], y2], '-', linewidth=1.2)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.set_xlim(ego_pos[0]-r_max-5, ego_pos[0]+r_max+5)
    ax.set_ylim(ego_pos[1]-r_max-5, ego_pos[1]+r_max+5)

def save_gif_from_frames(frame_fns, out_path='spart_scan.gif', fps=12):
    imgs = [imageio.v2.imread(fn) for fn in frame_fns]
    imageio.mimsave(out_path, imgs, fps=fps)
    print("Saved GIF:", out_path)    


def save_fig(fig, name):
    png_path = os.path.join(config["plots_dir"], name + ".png")
    fig.savefig(png_path, dpi=200, bbox_inches='tight')
    if config["save_plots_svg"]:
        svg_path = os.path.join(config["plots_dir"], name + ".svg")
        fig.savefig(svg_path, bbox_inches='tight')
    print("Saved:", png_path)

def plot_fps_vs_vehicles(df, df_results):

    fig, ax = plt.subplots(figsize=(8, 5))

    # SPART results
    df_results_f = df_results[df_results["N_vehicles"] != 10].copy()
    grouped_spart = (
        df_results_f
        .groupby("N_vehicles")
        .agg(time_ms=("time_ms", "mean"))
        .reset_index()
    )
    grouped_spart["FPS_mean"] = 1000.0 / grouped_spart["time_ms"]

    ax.plot(
        grouped_spart["N_vehicles"],
        grouped_spart["FPS_mean"],
        "-o",
        linewidth=2.5,
        markersize=6,
        label="SPART",
    )

    # Baseline results
    df = df.copy()
    df["time_ms"] = df["time_total_sec"] * 1000.0
    grouped = (
        df.groupby("N_vehicles")
        .agg(time_ms=("time_ms", "mean"))
        .reset_index()
    )
    grouped["FPS_mean"] = 1000.0 / grouped["time_ms"]

    ax.plot(
        grouped["N_vehicles"],
        grouped["FPS_mean"],
        "--s",
        linewidth=2.5,
        markersize=6,
        label="Vanila Raycasting",
    )

    ax.set_xlabel("Number of vehicles", fontweight="bold")
    ax.set_ylabel("FPS (mean, log scale)", fontweight="bold")
    ax.set_title("FPS with Number of Vehicles (SPART vs Vanilla Raycasting)", fontweight="bold")

    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(frameon=True)

    ax.set_yscale('log')
    ax.set_ylim(1e1, 1e5)


    save_fig(fig, "fps_vs_vehicles_comparison")
    plt.close(fig)

Loaded 8875 frames. Running frames 0..8873


### Vanilla RayCasting Execution

In [31]:

def run_vanilla_benchmark(synthetic=False):
    rng = np.random.RandomState(config["random_seed"])
    rows = []
    sample_images = []
    # dataset run
    if not synthetic:
        frame_list = [f for f in range(config["frame_start"], config["frame_end"]+1) if f in frames]
        for fid in frame_list:
            fv = frames[fid]          
            angles, ranges, hit_tracks, metrics = process_frame_vanilla_instrumented(
                fv, (995.0, 1000.0), 0.0, math.radians(config["fov_deg"]), math.radians(config["delta_deg"]),
                config["r_max"], enable_instrumentation=config["enable_instrumentation"])
            row = {
                "frame_id": fid,
                "timestamp_s": frame_timestamps.get(fid, float(fid)),
                "N_vehicles": metrics["n_vehicles"],
                "P_processed": metrics["n_vehicles"],  # vanilla processes all
                "S_segments": metrics["n_segments"],
                "K_beams": metrics["K_beams"],
                "sum_m_k": metrics["sum_m_k"],
                "num_hits": metrics["num_hits"],
                "time_build_sec": metrics["time_build_sec"],
                "time_total_sec": metrics["time_total_sec"],
                "memory_rss_mb": metrics["memory_rss_mb"],
                "engine": "vanilla"
            }
            rows.append(row)

    else:
        # synthetic density sweep
        for n in config["synthetic_densities"]:
            for trial in range(config["synthetic_repeats"]):
                cars = []
                for i in range(n):
                    ang = rng.uniform(-math.pi, math.pi)
                    rdist = rng.uniform(1.0, config["r_max"]*1.5)
                    cx = 995.0 + rdist * math.cos(ang)
                    cy = 1000.0 + rdist * math.sin(ang)
                    yaw = rng.uniform(-math.pi, math.pi)
                    cars.append({"track_id": i+1, "pos": np.array([cx, cy]), "vel": np.array([0.0,0.0]), "yaw": yaw, "L":4.0, "W":1.8})
                angles, ranges, hit_tracks, metrics = process_frame_vanilla_instrumented(
                    cars, (995.0,1000.0), 0.0, math.radians(config["fov_deg"]), math.radians(config["delta_deg"]),
                    config["r_max"], enable_instrumentation=config["enable_instrumentation"])
                row = {"frame_id": f"synthetic_{n}_{trial}", "timestamp_s": time.time(), "N_vehicles": n,
                       "P_processed": n, "S_segments": metrics["n_segments"], "K_beams": metrics["K_beams"],
                       "sum_m_k": metrics["sum_m_k"], "num_hits": metrics["num_hits"],
                       "time_build_sec": metrics["time_build_sec"], "time_total_sec": metrics["time_total_sec"],
                       "memory_rss_mb": metrics["memory_rss_mb"], "engine": "vanilla"}
                rows.append(row)
    # save_gif_from_frames(sample_images, out_path='output_vanilla/raycasting_metrics1.gif', fps=8)                
    df_out = pd.DataFrame(rows)
    df_out.to_csv(os.path.join(config["output_dir"], "per_frame_log_vanilla.csv"), index=False)
    print("Saved per-frame log to", os.path.join(config["output_dir"], "per_frame_log_vanilla.csv"))
    return df_out


In [32]:
# Run dataset vanilla benchmark
df_van = run_vanilla_benchmark(synthetic=False)

# Summary stats
summary = {
    "frames": len(df_van),
    "mean_time_ms": df_van["time_total_sec"].mean()*1000.0,
    "median_time_ms": df_van["time_total_sec"].median()*1000.0,
    "std_time_ms": df_van["time_total_sec"].std()*1000.0,
    "mean_sum_m_k": df_van["sum_m_k"].mean(),
    "mean_memory_mb": df_van["memory_rss_mb"].mean()
}
with open(os.path.join(config["output_dir"], "summary_report_vanilla.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Vanilla summary:", summary)


Saved per-frame log to OutputVanilla\per_frame_log_vanilla.csv
Vanilla summary: {'frames': 8749, 'mean_time_ms': np.float64(6.417632483768591), 'median_time_ms': np.float64(5.541299993637949), 'std_time_ms': np.float64(3.3248744478663737), 'mean_sum_m_k': np.float64(6511.5450908675275), 'mean_memory_mb': np.float64(2296.4852134886846)}


### Comparison Plot Generation

In [33]:
# Plots
plot_fps_vs_vehicles(df_van, df_results)
print("Plots saved to", config["plots_dir"])

Saved: OutputVanilla/plots\fps_vs_vehicles_comparison.png
Plots saved to OutputVanilla/plots


### Metric Computation

In [34]:

def compute_aggregates(df):
    df2 = df.copy()
    df2["M_total"] = df2["N_vehicles"] * 4
    df2["K_times_M"] = df2["K_beams"] * df2["M_total"]
    df2["SEI_frame"] = df2["K_times_M"] / (df2["sum_m_k"].replace(0, np.nan))
    agg = {
        "frames": len(df2),
        "mean_time_ms": df2["time_total_sec"].mean()*1000.0,
        "median_time_ms": df2["time_total_sec"].median()*1000.0,
        "fps_mean": 1.0 / df2["time_total_sec"].mean() if df2["time_total_sec"].mean() > 0 else float('nan'),
        "mean_sum_m_k": df2["sum_m_k"].mean(),
        "mean_SEI": df2["SEI_frame"].mean(skipna=True),
        "mean_eligibility_muted_ratio": ((df2["N_vehicles"] - df2["P_processed"]) / df2["N_vehicles"]).replace([np.inf, -np.inf], np.nan).mean(skipna=True)
    }
    return agg

agg = compute_aggregates(df_van)
with open(os.path.join(config["output_dir"], "summary_stats.json"), "w") as f:
    json.dump(agg, f, indent=2)
print("Aggregates saved. SEI mean:", agg.get("mean_SEI"))

Aggregates saved. SEI mean: 1.0
